<a href="https://colab.research.google.com/github/kousiknandy/pycolab/blob/main/LRU_caches.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
from collections import OrderedDict
import threading

class LRUCache:

    def __init__(self, capacity):
        self.capacity = capacity
        self.cache = OrderedDict()
        self.lock = threading.Lock()

    def get(self, key):
        with self.lock:
            if key not in self.cache: return None
            val = self.cache.get(key)
            self.cache.move_to_end(key)
        return val

    def put(self, key, val):
        with self.lock:
            if key not in self.cache and len(self.cache) >= self.capacity:
                self.cache.popitem(last=False)
            self.cache[key] = val
            self.cache.move_to_end(key)

In [36]:
print('--- Test Case 1: Basic LRU eviction ---')
cache1 = LRUCache(2)
cache1.put(1, 1)
cache1.put(2, 2)
# Cache is {1:1, 2:2}
print(f"Cache after put(1,1), put(2,2): {cache1.cache}")

# Accessing 1 makes it most recently used
cache1.get(1)
# Cache is {2:2, 1:1}
print(f"Cache after get(1): {cache1.cache}")

# Putting 3 evicts 2 (least recently used)
cache1.put(3, 3)
# Cache is {1:1, 3:3}
print(f"Cache after put(3,3): {cache1.cache}")
assert cache1.get(2) is None, "Test Case 1 Failed: Key 2 should have been evicted."
assert cache1.get(1) == 1, "Test Case 1 Failed: Key 1 should still be present."
assert cache1.get(3) == 3, "Test Case 1 Failed: Key 3 should be present."
print('Test Case 1 Passed.')

print('\n--- Test Case 2: Eviction order with multiple gets ---')
cache2 = LRUCache(3)
cache2.put(1, 10)
cache2.put(2, 20)
cache2.put(3, 30)
# Cache is {1:10, 2:20, 3:30}
print(f"Cache after put(1,10), put(2,20), put(3,30): {cache2.cache}")

cache2.get(1) # 1 is now MRU
# Cache is {2:20, 3:30, 1:10}
print(f"Cache after get(1): {cache2.cache}")

cache2.put(4, 40) # 2 should be evicted
# Cache is {3:30, 1:10, 4:40}
print(f"Cache after put(4,40): {cache2.cache}")

assert cache2.get(2) is None, "Test Case 2 Failed: Key 2 should have been evicted."
assert cache2.get(3) == 30, "Test Case 2 Failed: Key 3 should be present."
assert cache2.get(1) == 10, "Test Case 2 Failed: Key 1 should be present."
assert cache2.get(4) == 40, "Test Case 2 Failed: Key 4 should be present."
print('Test Case 2 Passed.')

print('\n--- Test Case 3: Update value of existing key ---')
cache3 = LRUCache(2)
cache3.put(1, 100)
cache3.put(2, 200)
# Cache is {1:100, 2:200}
print(f"Cache after put(1,100), put(2,200): {cache3.cache}")

cache3.put(1, 101) # Update 1, making it MRU
# Cache is {2:200, 1:101}
print(f"Cache after put(1,101): {cache3.cache}")

cache3.put(3, 300) # 2 should be evicted
# Cache is {1:101, 3:300}
print(f"Cache after put(3,300): {cache3.cache}")

assert cache3.get(2) is None, "Test Case 3 Failed: Key 2 should have been evicted."
assert cache3.get(1) == 101, "Test Case 3 Failed: Key 1 should be present with updated value."
assert cache3.get(3) == 300, "Test Case 3 Failed: Key 3 should be present."
print('Test Case 3 Passed.')

--- Test Case 1: Basic LRU eviction ---
Cache after put(1,1), put(2,2): OrderedDict({1: 1, 2: 2})
Cache after get(1): OrderedDict({2: 2, 1: 1})
Cache after put(3,3): OrderedDict({1: 1, 3: 3})
Test Case 1 Passed.

--- Test Case 2: Eviction order with multiple gets ---
Cache after put(1,10), put(2,20), put(3,30): OrderedDict({1: 10, 2: 20, 3: 30})
Cache after get(1): OrderedDict({2: 20, 3: 30, 1: 10})
Cache after put(4,40): OrderedDict({3: 30, 1: 10, 4: 40})
Test Case 2 Passed.

--- Test Case 3: Update value of existing key ---
Cache after put(1,100), put(2,200): OrderedDict({1: 100, 2: 200})
Cache after put(1,101): OrderedDict({2: 200, 1: 101})
Cache after put(3,300): OrderedDict({1: 101, 3: 300})
Test Case 3 Passed.


In [44]:
class LRUDecorator(LRUCache):

    def __init__(self, func, capacity):
        super().__init__(capacity)
        self.func = func

    def __call__(self, *args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        val = self.get(key)
        if val is not None: return val
        val = self.func(*args, **kwargs)
        self.put(key, val)
        return val

In [38]:
print('\n--- Test Cases for LRUDecorator ---')

# A simple function to be decorated
def expensive_calculation(a, b, c=0):
    print(f"  Calling expensive_calculation({a}, {b}, c={c})")
    return a * b + c

# Test Case 1: Basic Caching
print('\n--- Test Case 1: Basic LRUDecorator Caching ---')
cached_calc1 = LRUDecorator(expensive_calculation, capacity=2)

# First call, should execute function
result1_1 = cached_calc1(1, 2)
print(f"Result 1,2: {result1_1}")
assert result1_1 == 2, "Test Case 1 Failed: Initial call incorrect."

# Second call with same args, should use cache
result1_2 = cached_calc1(1, 2)
print(f"Result 1,2 (cached): {result1_2}")
assert result1_2 == 2, "Test Case 1 Failed: Cached call incorrect."

# Different args, should execute function
result1_3 = cached_calc1(3, 4)
print(f"Result 3,4: {result1_3}")
assert result1_3 == 12, "Test Case 1 Failed: Second unique call incorrect."

# Verify cache state
assert ( ( (1, 2), tuple(sorted({}.items())) ) ) in cached_calc1.cache, "Test Case 1 Failed: Key (1,2) not in cache."
assert ( ( (3, 4), tuple(sorted({}.items())) ) ) in cached_calc1.cache, "Test Case 1 Failed: Key (3,4) not in cache."
print('Test Case 1 Passed.')


# Test Case 2: LRU Eviction with Decorator
print('\n--- Test Case 2: LRUDecorator Eviction ---')
cached_calc2 = LRUDecorator(expensive_calculation, capacity=2)

_ = cached_calc2(10, 1) # Cache: { ((10,1),()):10 }
_ = cached_calc2(20, 2) # Cache: { ((10,1),()):10, ((20,2),()):40 } (10,1) is LRU
print(f"Cache state after two puts: {cached_calc2.cache.keys()}")

_ = cached_calc2(10, 1) # Access (10,1), making it MRU. Cache: { ((20,2),()):40, ((10,1),()):10 } (20,2) is LRU
print(f"Cache state after accessing (10,1): {cached_calc2.cache.keys()}")

_ = cached_calc2(30, 3) # New item (30,3), (20,2) should be evicted (LRU)
# Cache: { ((10,1),()):10, ((30,3),()):90 } (10,1) is LRU
print(f"Cache state after putting (30,3): {cached_calc2.cache.keys()}")

# Try to get (20,2) - should not be in cache, will re-calculate and evict (10,1)
result2_1 = cached_calc2(20, 2)
print(f"Result 20,2 (after eviction and re-calc): {result2_1}")
assert result2_1 == 40, "Test Case 2 Failed: Evicted item (20,2) should be re-calculated."
assert ( ( (20, 2), tuple(sorted({}.items())) ) ) in cached_calc2.cache, "Test Case 2 Failed: Re-calculated item (20,2) not in cache."
assert ( ( (10, 1), tuple(sorted({}.items())) ) ) not in cached_calc2.cache, "Test Case 2 Failed: Key (10,1) should have been evicted."
# Current cache: {((30,3),()):90, ((20,2),()):40} (30,3) is LRU

# Now add (40,4). This should evict (30,3).
_ = cached_calc2(40, 4) # Cache: { ((20,2),()):40, ((40,4),()):160 } (20,2) is LRU
print(f"Cache state after putting (40,4): {cached_calc2.cache.keys()}")

# Access (30,3) - should not be in cache, will re-calculate and evict (20,2).
result2_2 = cached_calc2(30, 3)
print(f"Result 30,3 (after eviction and re-calc): {result2_2}")
assert result2_2 == 90, "Test Case 2 Failed: Evicted item (30,3) should be re-calculated."
assert ( ( (30, 3), tuple(sorted({}.items())) ) ) in cached_calc2.cache, "Test Case 2 Failed: Re-calculated item (30,3) not in cache."
assert ( ( (20, 2), tuple(sorted({}.items())) ) ) not in cached_calc2.cache, "Test Case 2 Failed: Key (20,2) should have been evicted."
print('Test Case 2 Passed.')


# Test Case 3: Keyword Arguments and Updates
print('\n--- Test Case 3: LRUDecorator with Keyword Args ---')
cached_calc3 = LRUDecorator(expensive_calculation, capacity=2)

_ = cached_calc3(5, 5, c=1) # Cache: { ((5,5),('c',1)):26 }
_ = cached_calc3(6, 6)     # Cache: { ((5,5),('c',1)):26, ((6,6),()):36 } (5,5,c=1) is LRU
print(f"Cache state after two puts: {cached_calc3.cache.keys()}")

_ = cached_calc3(5, 5, c=1) # Access, makes it MRU. Cache: { ((6,6),()):36, ((5,5),('c',1)):26 } (6,6) is LRU
print(f"Cache state after accessing (5,5, c=1): {cached_calc3.cache.keys()}")

_ = cached_calc3(7, 7, c=2) # New item, evicts ((6,6),()). Cache: { ((5,5),('c',1)):26, ((7,7),('c',2)):51 } (5,5,c=1) is LRU
print(f"Cache state after putting (7,7, c=2): {cached_calc3.cache.keys()}")

assert cached_calc3.get(((6, 6), tuple(sorted({}.items())))) is None, "Test Case 3 Failed: Key (6,6) should have been evicted."
assert cached_calc3(5, 5, c=1) == 26, "Test Case 3 Failed: Key (5,5, c=1) should be present."
assert cached_calc3(7, 7, c=2) == 51, "Test Case 3 Failed: Key (7,7, c=2) should be present."
print('Test Case 3 Passed.')


--- Test Cases for LRUDecorator ---

--- Test Case 1: Basic LRUDecorator Caching ---
  Calling expensive_calculation(1, 2, c=0)
Result 1,2: 2
Result 1,2 (cached): 2
  Calling expensive_calculation(3, 4, c=0)
Result 3,4: 12
Test Case 1 Passed.

--- Test Case 2: LRUDecorator Eviction ---
  Calling expensive_calculation(10, 1, c=0)
  Calling expensive_calculation(20, 2, c=0)
Cache state after two puts: odict_keys([((10, 1), ()), ((20, 2), ())])
Cache state after accessing (10,1): odict_keys([((20, 2), ()), ((10, 1), ())])
  Calling expensive_calculation(30, 3, c=0)
Cache state after putting (30,3): odict_keys([((10, 1), ()), ((30, 3), ())])
  Calling expensive_calculation(20, 2, c=0)
Result 20,2 (after eviction and re-calc): 40
  Calling expensive_calculation(40, 4, c=0)
Cache state after putting (40,4): odict_keys([((20, 2), ()), ((40, 4), ())])
  Calling expensive_calculation(30, 3, c=0)
Result 30,3 (after eviction and re-calc): 90
Test Case 2 Passed.

--- Test Case 3: LRUDecorator wit

In [41]:
import heapq
import time

class LRUTTL(LRUCache):

    def __init__(self, capacity):
        super().__init__(capacity)
        self.timebombs = []

    def get(self, key):
        now = time.time()
        val = super().get(key)
        if val is None: return None
        # Corrected: If expiry_time (val[1]) is less than current time (now), it has expired.
        if val[1] < now:
            del self.cache[key] # Remove the expired item from the cache
            return None
        return val[0] # Return the actual value, not the (value, expiry_time) tuple

    def put(self, key, val, ttl):
        now = time.time()
        # Clean up any expired items from timebombs before adding a new one
        while self.timebombs and self.timebombs[0][0] < now:
            _, expired_key = heapq.heappop(self.timebombs)
            # Only delete if the key still exists and hasn't been updated with a new TTL
            # or evicted by LRU already. We check val[1] to ensure it's the same entry.
            if expired_key in self.cache and self.cache[expired_key][1] < now:
                 del self.cache[expired_key]

        # If adding a new item would exceed capacity, evict LRU first
        # This logic is handled by super().put() for LRUCache
        super().put(key, (val, now + ttl))
        heapq.heappush(self.timebombs, (now + ttl, key))

In [42]:
print('\n--- Test Case for LRUTTL: Time-based Eviction ---')

# The simple_function is no longer directly used for LRUTTL as it's not a decorator
def simple_function_placeholder(x):
    return x * 10

# Initialize LRUTTL without 'func' argument as it now inherits directly from LRUCache
cached_ttl = LRUTTL(capacity=2)

# Put an item with a 0.5-second TTL
print('Putting key 1 with value 10 and TTL of 0.5 seconds.')
cached_ttl.put(1, 10, 0.5)
assert cached_ttl.get(1) == 10, "Test 1 Failed: Key 1 should be present immediately after put."
print(f"Cache state after put(1): {cached_ttl.cache.keys()}")

# Wait for less than TTL
time.sleep(0.3)
assert cached_ttl.get(1) == 10, "Test 1 Failed: Key 1 should still be present before TTL expires."
print(f"Cache state after 0.3s sleep: {cached_ttl.cache.keys()}")

# Wait for TTL to expire
time.sleep(0.3) # Total sleep is 0.6s (0.3 + 0.3)
print('Waiting for 0.3 more seconds (total 0.6s).')
assert cached_ttl.get(1) is None, "Test 1 Failed: Key 1 should have been evicted after TTL expired."
print(f"Cache state after TTL expired: {cached_ttl.cache.keys()}")

print('\n--- Test Case for LRUTTL: Combined LRU and TTL Eviction ---')

# Reset cache for new test
cached_ttl_combo = LRUTTL(capacity=2)

# Put items with different TTLs
cached_ttl_combo.put(1, 10, 1.0) # Will expire later
cached_ttl_combo.put(2, 20, 0.2) # Will expire soon
print(f"Cache state after put(1,10,1.0) and put(2,20,0.2): {cached_ttl_combo.cache.keys()}")

# Access 1 to make it MRU
cached_ttl_combo.get(1)
print(f"Cache state after get(1): {cached_ttl_combo.cache.keys()}")

# Wait for key 2 to expire
time.sleep(0.3) # Key 2 should expire
print('Waiting 0.3 seconds for key 2 to expire.')

# Put a new item, which should evict key 1 (LRU if key 2 is already removed)
# Calling put will also trigger a timebomb check which should remove expired key 2
cached_ttl_combo.put(3, 30, 0.5)
print(f"Cache state after put(3,30,0.5): {cached_ttl_combo.cache.keys()}")

# Check if key 2 was evicted (due to TTL) and key 1 should still be present
assert cached_ttl_combo.get(2) is None, "Test Combo Failed: Key 2 should have expired."
assert cached_ttl_combo.get(1) == 10, "Test Combo Failed: Key 1 should still be present."
assert cached_ttl_combo.get(3) == 30, "Test Combo Failed: Key 3 should be present."

# Now, access 3 and put 4. 3 should be MRU. 4 is new. Cache will be {3,4}
_ = cached_ttl_combo.get(3)
cached_ttl_combo.put(4, 40, 0.5)
print(f"Cache state after get(3) and put(4,40,0.5): {cached_ttl_combo.cache.keys()}")
assert cached_ttl_combo.get(3) == 30, "Test Combo Failed: Key 3 should be present."
assert cached_ttl_combo.get(4) == 40, "Test Combo Failed: Key 4 should be present."

print('LRUTTL Test Cases Passed.')


--- Test Case for LRUTTL: Time-based Eviction ---
Putting key 1 with value 10 and TTL of 0.5 seconds.
Cache state after put(1): odict_keys([1])
Cache state after 0.3s sleep: odict_keys([1])
Waiting for 0.3 more seconds (total 0.6s).
Cache state after TTL expired: odict_keys([])

--- Test Case for LRUTTL: Combined LRU and TTL Eviction ---
Cache state after put(1,10,1.0) and put(2,20,0.2): odict_keys([1, 2])
Cache state after get(1): odict_keys([2, 1])
Waiting 0.3 seconds for key 2 to expire.
Cache state after put(3,30,0.5): odict_keys([1, 3])
Cache state after get(3) and put(4,40,0.5): odict_keys([3, 4])
LRUTTL Test Cases Passed.
